# Experiment 5.2.1 — FF Multi-Tau Memory × Objective

Analysis-only notebook. It reads finalized artifacts produced by the multi-CPU experiment runner. It never trains models, launches Slurm, regenerates missing runs, or selects a condition using test BA.

Primary comparison: `SingleTau vs MultiTau` × `Endpoint CE vs Endpoint + Fixed250 auxiliary CE`. Final deployed readout is always hidden `Uend -> Linear -> logits`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate writingRing repository root")


REPO_ROOT = find_repo_root()
ARTIFACT_ROOT = (
    REPO_ROOT
    / "notebooks"
    / "artifacts"
    / "experiment_5_2_1_ff_multitau_objectives"
    / "frozen_exp3_l2_ff_multitau_objectives_v1"
)
runs = pd.read_csv(ARTIFACT_ROOT / "runs.csv")
histories = pd.read_csv(ARTIFACT_ROOT / "histories.csv")
local_reference = pd.read_csv(ARTIFACT_ROOT / "local_reference.csv")
manifest = json.loads((ARTIFACT_ROOT / "manifest.json").read_text(encoding="utf-8"))

assert len(runs) == manifest["expected_runs"] == 20
assert set(runs["seed"]) == set(manifest["seeds"])
runs.head()


## Condition summary

Checkpoint selection is per-run validation-only. Test metrics below are descriptive evaluation of those already-selected checkpoints.


In [ ]:
summary_metrics = [
    "native_val_balanced_accuracy",
    "native_test_balanced_accuracy",
    "hidden_uend_val_ba",
    "hidden_uend_test_ba",
    "hidden_fixed250_ordered_test_ba",
    "hidden_relative10_ordered_test_ba",
    "hidden_whole_count_test_ba",
]
condition_summary = (
    runs.groupby(["memory_mode", "objective"])[summary_metrics]
    .agg(["mean", "std"])
    .sort_values(("native_val_balanced_accuracy", "mean"), ascending=False)
)
condition_summary


In [ ]:
diagnostic_runs = runs.copy()
diagnostic_runs["head_utilization_gap"] = (
    diagnostic_runs["hidden_uend_test_ba"]
    - diagnostic_runs["native_test_balanced_accuracy"]
)
diagnostic_runs["trajectory_endpoint_gap"] = (
    diagnostic_runs["hidden_relative10_ordered_test_ba"]
    - diagnostic_runs["hidden_uend_test_ba"]
)
diagnostic_runs["local_fixed250_gap"] = np.nan

local_fixed = (
    local_reference[local_reference["probe_type"] == "local_fixed250_ordered"]
    .set_index("seed")["test_ba"]
)
diagnostic_runs["local_fixed250_gap"] = diagnostic_runs.apply(
    lambda row: local_fixed.loc[row["seed"]] - row["hidden_uend_test_ba"], axis=1
)

diagnostic_summary = (
    diagnostic_runs.groupby(["memory_mode", "objective"])[
        ["head_utilization_gap", "trajectory_endpoint_gap", "local_fixed250_gap"]
    ]
    .agg(["mean", "std"])
)
diagnostic_summary


## Paired factorial effects

All effects are paired by seed. `memory_effect` is MultiTau − SingleTau within the same objective. `objective_effect` is auxiliary objective − endpoint-only objective within the same memory mode. `interaction_effect` is the difference-of-differences.


In [ ]:
def paired_memory_effect(frame, metric):
    wide = frame.pivot(index="seed", columns=["objective", "memory_mode"], values=metric)
    rows = []
    for objective in manifest["objectives"]:
        effect = wide[(objective, "multitau_4567")] - wide[(objective, "single_shift7")]
        rows.append({
            "objective": objective,
            "metric": metric,
            "mean": effect.mean(),
            "sd": effect.std(ddof=1),
            "positive_seeds": int((effect > 0).sum()),
        })
    return pd.DataFrame(rows)


def paired_objective_effect(frame, metric):
    wide = frame.pivot(index="seed", columns=["memory_mode", "objective"], values=metric)
    rows = []
    for memory_mode in manifest["memory_modes"]:
        effect = (
            wide[(memory_mode, "endpoint_fixed250_aux_ce")]
            - wide[(memory_mode, "endpoint_ce")]
        )
        rows.append({
            "memory_mode": memory_mode,
            "metric": metric,
            "mean": effect.mean(),
            "sd": effect.std(ddof=1),
            "positive_seeds": int((effect > 0).sum()),
        })
    return pd.DataFrame(rows)


def paired_interaction_effect(frame, metric):
    wide = frame.pivot(index="seed", columns=["memory_mode", "objective"], values=metric)
    memory_old = (
        wide[("multitau_4567", "endpoint_ce")]
        - wide[("single_shift7", "endpoint_ce")]
    )
    memory_new = (
        wide[("multitau_4567", "endpoint_fixed250_aux_ce")]
        - wide[("single_shift7", "endpoint_fixed250_aux_ce")]
    )
    interaction = memory_new - memory_old
    return pd.DataFrame({
        "seed": interaction.index,
        "metric": metric,
        "interaction_effect": interaction.values,
    })

factorial_metrics = [
    "native_val_balanced_accuracy",
    "native_test_balanced_accuracy",
    "hidden_uend_test_ba",
    "hidden_relative10_ordered_test_ba",
]
memory_effect = pd.concat(
    [paired_memory_effect(runs, metric) for metric in factorial_metrics], ignore_index=True
)
objective_effect = pd.concat(
    [paired_objective_effect(runs, metric) for metric in factorial_metrics], ignore_index=True
)
interaction_effect = pd.concat(
    [paired_interaction_effect(runs, metric) for metric in factorial_metrics], ignore_index=True
)
interaction_summary = interaction_effect.groupby("metric")["interaction_effect"].agg(["mean", "std"])

memory_effect, objective_effect, interaction_summary


## Validation-primary condition view


In [ ]:
plot_summary = (
    runs.groupby(["memory_mode", "objective"])
    .agg(
        val_mean=("native_val_balanced_accuracy", "mean"),
        val_sd=("native_val_balanced_accuracy", "std"),
        test_mean=("native_test_balanced_accuracy", "mean"),
        test_sd=("native_test_balanced_accuracy", "std"),
    )
    .reset_index()
)
plot_summary["condition"] = plot_summary["memory_mode"] + "\n" + plot_summary["objective"]

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(plot_summary))
ax.errorbar(x - 0.08, plot_summary["val_mean"], yerr=plot_summary["val_sd"], fmt="o", capsize=4, label="Validation BA")
ax.errorbar(x + 0.08, plot_summary["test_mean"], yerr=plot_summary["test_sd"], fmt="o", capsize=4, label="Test BA")
ax.set_xticks(x)
ax.set_xticklabels(plot_summary["condition"], rotation=20, ha="right")
ax.set_ylabel("Balanced accuracy")
ax.set_title("Exp5.2.1 native endpoint classifier")
ax.legend()
fig.tight_layout()
plt.show()


## Representation and endpoint-head diagnostics


In [ ]:
probe_summary = (
    diagnostic_runs.groupby(["memory_mode", "objective"])
    .agg(
        whole=("hidden_whole_count_test_ba", "mean"),
        fixed250=("hidden_fixed250_ordered_test_ba", "mean"),
        relative10=("hidden_relative10_ordered_test_ba", "mean"),
        uend_probe=("hidden_uend_test_ba", "mean"),
        native=("native_test_balanced_accuracy", "mean"),
        head_gap=("head_utilization_gap", "mean"),
        trajectory_endpoint_gap=("trajectory_endpoint_gap", "mean"),
    )
    .reset_index()
)
probe_summary


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
labels = [f"{row.memory_mode}\n{row.objective}" for row in probe_summary.itertuples()]
x = np.arange(len(labels))
for metric, offset in [("relative10", -0.15), ("uend_probe", 0.0), ("native", 0.15)]:
    ax.scatter(x + offset, probe_summary[metric], label=metric)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Test balanced accuracy")
ax.set_title("Hidden trajectory -> Uend -> native head")
ax.legend()
fig.tight_layout()
plt.show()


## Validation learning curves


In [ ]:
curve = (
    histories.groupby(["memory_mode", "objective", "epoch"])["val_balanced_accuracy"]
    .agg(["mean", "std"])
    .reset_index()
)
fig, ax = plt.subplots(figsize=(10.5, 5.5))
for (memory_mode, objective), group in curve.groupby(["memory_mode", "objective"]):
    group = group.sort_values("epoch")
    label = f"{memory_mode} | {objective}"
    ax.plot(group["epoch"], group["mean"], label=label)
    ax.fill_between(group["epoch"], group["mean"] - group["std"], group["mean"] + group["std"], alpha=0.15)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation balanced accuracy")
ax.set_title("Exp5.2.1 validation learning curves")
ax.legend()
fig.tight_layout()
plt.show()


## Frozen Local reference


In [ ]:
local_summary = (
    local_reference.groupby("probe_type")["test_ba"]
    .agg(["mean", "std"])
    .sort_values("mean")
)
local_summary
